In [0]:
def test_bronze_table_exists(spark):

    df = spark.table("socialmedia_databricks.bronze.bronze_valid_tweets")

    assert df.count() > 0

In [0]:
# ==========================================================
# Project      : Social Media Sentiment Analysis
# Notebook     : test_bronze
# Description  : Bronze Layer Testing
# ==========================================================

tables = [

    "bronze_valid_tweets",
    "bronze_tweets",
    "bronze_sentiment1",
    "bronze_trends",
    "bronze_user_metadata"

]

In [0]:
# ==========================================================
# Test 1 : Bronze Tables Exist
# ==========================================================

for table in tables:

    assert spark.catalog.tableExists(
        f"socialmedia_databricks.bronze.{table}"
    ), f"{table} does not exist"

print("✅ Test 1 Passed : All Bronze tables exist")

✅ Test 1 Passed : All Bronze tables exist


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
print(spark.version)

3.5.0


In [0]:
import pytest

print(pytest.__version__)

9.1.1


In [0]:
# ==========================================================
# Test 2 : Row Count Validation
# ==========================================================

for table in tables:

    row_count = spark.table(
        f"socialmedia_databricks.bronze.{table}"
    ).count()

    assert row_count > 0, f"{table} is empty"

print("✅ Test 2 Passed : All Bronze tables contain data")

✅ Test 2 Passed : All Bronze tables contain data


In [0]:
# ==========================================================
# Test 3 : Required Columns Validation
# Bronze Valid Tweets
# ==========================================================

required_columns = [
    "tweet_id",
    "topic_category",
    "tweet_text",
    "tweet_timestamp",
    "impressions",
    "likes",
    "retweets",
    "replies",
    "engagement_count",
    "sentiment_score",
    "bronze_load_time",
    "pipeline_name",
    "source_system",
    "ingestion_date"
]

df = spark.table("socialmedia_databricks.bronze.bronze_valid_tweets")

for column in required_columns:
    assert column in df.columns, f"{column} is missing"

print("✅ Test 3 Passed : bronze_valid_tweets columns validated")

✅ Test 3 Passed : bronze_valid_tweets columns validated


In [0]:
# ==========================================================
# Test 4 : Null Analysis
# Bronze Valid Tweets
# ==========================================================

from pyspark.sql.functions import col

df = spark.table("socialmedia_databricks.bronze.bronze_valid_tweets")

critical_columns = [
    "tweet_id",
    "tweet_text",
    "tweet_timestamp"
]

for column in critical_columns:

    null_count = df.filter(col(column).isNull()).count()

    print(f"{column} -> NULL Count : {null_count}")

print("✅ Test 4 Passed : Null analysis completed")

tweet_id -> NULL Count : 0
tweet_text -> NULL Count : 0
tweet_timestamp -> NULL Count : 247
✅ Test 4 Passed : Null analysis completed


In [0]:
# ==========================================================
# Test 5 : Duplicate Tweet ID Analysis
# ==========================================================

from pyspark.sql.functions import col

df = spark.table("socialmedia_databricks.bronze.bronze_valid_tweets")

duplicate_count = (
    df.groupBy("tweet_id")
      .count()
      .filter(col("count") > 1)
      .count()
)

print(f"Duplicate tweet_id records : {duplicate_count}")

print("✅ Test 5 Passed : Duplicate analysis completed")

Duplicate tweet_id records : 253
✅ Test 5 Passed : Duplicate analysis completed


In [0]:
# ==========================================================
# Test 6 : Metadata Columns Validation
# ==========================================================

metadata_columns = [

    "bronze_load_time",
    "pipeline_name",
    "source_system",
    "ingestion_date"

]

for table in tables:

    df = spark.table(f"socialmedia_databricks.bronze.{table}")

    for column in metadata_columns:

        assert column in df.columns, f"{column} missing in {table}"

print("✅ Test 6 Passed : Metadata columns validated")

✅ Test 6 Passed : Metadata columns validated


In [0]:
# ==========================================================
# Test 7 : Metadata Values Validation
# ==========================================================

from pyspark.sql.functions import col

df = spark.table("socialmedia_databricks.bronze.bronze_valid_tweets")

assert df.filter(col("bronze_load_time").isNull()).count() == 0
assert df.filter(col("pipeline_name").isNull()).count() == 0
assert df.filter(col("source_system").isNull()).count() == 0
assert df.filter(col("ingestion_date").isNull()).count() == 0

print("✅ Test 7 Passed : Metadata values validated")

✅ Test 7 Passed : Metadata values validated


In [0]:
# ==========================================================
# Test 8 : Data Type Validation
# ==========================================================

df = spark.table("socialmedia_databricks.bronze.bronze_valid_tweets")

expected_datatypes = {

    "tweet_id": "string",
    "tweet_text": "string",
    "tweet_timestamp": "timestamp",
    "impressions": "double",
    "likes": "double",
    "retweets": "double",
    "replies": "double",
    "engagement_count": "double",
    "sentiment_score": "double"

}

actual_datatypes = dict(df.dtypes)

for column, datatype in expected_datatypes.items():

    assert actual_datatypes[column] == datatype, \
        f"{column} should be {datatype}, but found {actual_datatypes[column]}"

print("✅ Test 8 Passed : Data types validated")

✅ Test 8 Passed : Data types validated
